# Full re-evaluation for the FHN-FNO paper (Tables 1-5)

Reproduces all five paper tables under one training recipe (batch 256, cosine LR, 40 epochs, 5 seeds). Results are saved as JSON to Drive, along with checkpoints and a ready-to-paste tables.tex.

Expected Drive layout:
```
MyDrive/fhn_full_reeval/
    fhn_1d_8000.h5     <- upload your dataset here
    (results land here)
```

Budget: about 4 hours on an A100, so a Pro+ runtime is recommended. Resumable: each cell saves its output, so rerun cells 1-3 plus whichever one you want.

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Install and mount Drive

In [ ]:
!pip install -q h5py
from google.colab import drive
drive.mount('/content/drive')

## Locate data and output folder

Edit DRIVE_FOLDER if your data lives elsewhere.

In [ ]:
import os, shutil
DRIVE_FOLDER = '/content/drive/MyDrive/fhn_full_reeval'
DATASET_NAME = 'fhn_1d_8000.h5'
os.makedirs(DRIVE_FOLDER, exist_ok=True)
os.makedirs(os.path.join(DRIVE_FOLDER, 'checkpoints'), exist_ok=True)
DRIVE_DATA = os.path.join(DRIVE_FOLDER, DATASET_NAME)
LOCAL_DATA = f'/content/{DATASET_NAME}'
assert os.path.exists(DRIVE_DATA), f'Upload {DATASET_NAME} to {DRIVE_FOLDER} first.'
if not os.path.exists(LOCAL_DATA):
    print('Copying dataset to local disk for fast I/O...')
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
print(f'Dataset: {LOCAL_DATA} ({os.path.getsize(LOCAL_DATA)/1e9:.2f} GB)')
print(f'Outputs: {DRIVE_FOLDER}')

## Models and DataConfig

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DataConfig:
    Du_range  = (0.01, 0.1)
    Dv_range  = (0.005, 0.05)
    a_range   = (-0.1, 0.1)
    b_range   = (0.1, 0.5)
    tau_range = (1.0, 20.0)

class SpectralConv1d(nn.Module):
    def __init__(self, in_c, out_c, modes):
        super().__init__()
        self.in_c, self.out_c, self.modes = in_c, out_c, modes
        scale = 1 / (in_c * out_c)
        self.weights = nn.Parameter(scale * torch.randn(in_c, out_c, modes, dtype=torch.cfloat))
    def forward(self, x):
        B, _, nx = x.shape
        xf = torch.fft.rfft(x, dim=-1)
        out = torch.zeros(B, self.out_c, nx // 2 + 1, dtype=torch.cfloat, device=x.device)
        out[:, :, :self.modes] = torch.einsum('bix,iox->box', xf[:, :, :self.modes], self.weights)
        return torch.fft.irfft(out, n=nx, dim=-1)

class FourierLayer(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.w = nn.Conv1d(width, width, 1)
        self.norm = nn.InstanceNorm1d(width, affine=True)
        self.residual_weight = nn.Parameter(torch.ones(1))
    def forward(self, x):
        residual = x
        out = F.gelu(self.spectral(x) + self.w(x))
        out = self.norm(out)
        return out + self.residual_weight * residual

class ParameterEncoder(nn.Module):
    def __init__(self, n_params, width):
        super().__init__()
        h = width * 2
        self.encoder = nn.Sequential(
            nn.Linear(n_params, h), nn.GELU(), nn.LayerNorm(h),
            nn.Linear(h, width), nn.GELU(),
            nn.Linear(width, width), nn.LayerNorm(width))
    def forward(self, p): return self.encoder(p)

# Parametric FNO with FiLM conditioning
class ParametricFNO(nn.Module):
    def __init__(self, modes=16, width=64, n_layers=6, n_params=5):
        super().__init__()
        self.modes, self.width, self.n_layers = modes, width, n_layers
        self.param_encoder = ParameterEncoder(n_params, width)
        self.gamma_h = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.beta_h  = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.lift = nn.Sequential(
            nn.Conv1d(2, width * 2, 1), nn.GELU(),
            nn.Conv1d(width * 2, width, 1))
        self.proj = nn.Sequential(
            nn.Conv1d(width, width, 1), nn.GELU(),
            nn.Conv1d(width, 2, 1))
        self.layers = nn.ModuleList([FourierLayer(width, modes) for _ in range(n_layers)])
        self.global_residual = nn.Parameter(torch.ones(1) * 0.1)
    def forward(self, x, params):
        x0 = x.clone()
        x = self.lift(x)
        pf = self.param_encoder(params)
        for i, layer in enumerate(self.layers):
            x = layer(x)
            g = self.gamma_h[i](pf).unsqueeze(-1)
            b = self.beta_h[i](pf).unsqueeze(-1)
            x = g * x + b
        return self.proj(x) + self.global_residual * x0

# Broadcast FNO: params concatenated as constant spatial channels
class BroadcastFNO(nn.Module):
    def __init__(self, modes=16, width=64, n_layers=6, n_params=5):
        super().__init__()
        self.modes, self.width, self.n_layers = modes, width, n_layers
        in_c = 2 + n_params  # u, v plus broadcast lambda channels
        self.lift = nn.Sequential(
            nn.Conv1d(in_c, width * 2, 1), nn.GELU(),
            nn.Conv1d(width * 2, width, 1))
        self.proj = nn.Sequential(
            nn.Conv1d(width, width, 1), nn.GELU(),
            nn.Conv1d(width, 2, 1))
        self.layers = nn.ModuleList([FourierLayer(width, modes) for _ in range(n_layers)])
        self.global_residual = nn.Parameter(torch.ones(1) * 0.1)
    def forward(self, x, params):
        # broadcast params to (B, 5, nx) and concat
        B, _, nx = x.shape
        p_broadcast = params.unsqueeze(-1).expand(B, params.shape[1], nx)
        x_in = torch.cat([x, p_broadcast], dim=1)
        x0 = x.clone()  # only u, v for the residual
        z = self.lift(x_in)
        for layer in self.layers:
            z = layer(z)
        return self.proj(z) + self.global_residual * x0

# DeepONet baseline
class DeepONet(nn.Module):
    """Branch-trunk DeepONet mapping (u_t, v_t, lambda) to (u_{t+1}, v_{t+1})."""
    def __init__(self, nx=256, n_params=5, latent=128, branch_hidden=256, trunk_hidden=128, n_layers=4):
        super().__init__()
        self.nx, self.n_params, self.latent = nx, n_params, latent
        in_branch = 2 * nx + n_params
        branch = [nn.Linear(in_branch, branch_hidden), nn.GELU()]
        for _ in range(n_layers - 2):
            branch += [nn.Linear(branch_hidden, branch_hidden), nn.GELU()]
        branch += [nn.Linear(branch_hidden, 2 * latent)]  # 2 outputs (u, v)
        self.branch = nn.Sequential(*branch)
        trunk = [nn.Linear(1, trunk_hidden), nn.GELU()]
        for _ in range(n_layers - 2):
            trunk += [nn.Linear(trunk_hidden, trunk_hidden), nn.GELU()]
        trunk += [nn.Linear(trunk_hidden, 2 * latent)]
        self.trunk = nn.Sequential(*trunk)
        # cached spatial grid
        self.register_buffer('x_grid', torch.linspace(0, 1, nx).unsqueeze(-1))  # (nx, 1)
        self.global_residual = nn.Parameter(torch.ones(1) * 0.1)
    def forward(self, x, params):
        B = x.shape[0]
        b_in = torch.cat([x.flatten(1), params], dim=-1)  # (B, 2*nx + 5)
        b_out = self.branch(b_in)               # (B, 2*latent)
        t_out = self.trunk(self.x_grid)         # (nx, 2*latent)
        b_u, b_v = b_out[:, :self.latent], b_out[:, self.latent:]
        t_u, t_v = t_out[:, :self.latent], t_out[:, self.latent:]
        u = b_u @ t_u.T  # (B, nx)
        v = b_v @ t_v.T  # (B, nx)
        out = torch.stack([u, v], dim=1)  # (B, 2, nx)
        return out + self.global_residual * x

for cls, kwargs in [(ParametricFNO, {}), (BroadcastFNO, {}), (DeepONet, {})]:
    m = cls(**kwargs) if cls is not DeepONet else cls()
    n = sum(p.numel() for p in m.parameters())
    print(f'{cls.__name__:18s} {n:>10,d} params')
del m

## Dataset and eval metrics

Per-sample rel-L2 is averaged across val samples (matches Table 1). MSE and MAE are per-element. R^2 and Pearson are computed jointly across all samples.

In [ ]:
import h5py, json, time
import numpy as np
from torch.utils.data import Dataset, DataLoader

class H5SingleStepDataset(Dataset):
    def __init__(self, h5_path, indices, um, us, vm, vs, pm, ps):
        with h5py.File(h5_path, 'r') as f:
            self.u = np.array(f['u_traj'][indices])
            self.v = np.array(f['v_traj'][indices])
            self.p = np.array(f['params'][indices])
        self.um, self.us, self.vm, self.vs = um, us, vm, vs
        self.pm, self.ps = pm, ps
        self.n_samples, self.n_times, self.nx = self.u.shape
    def __len__(self):
        return self.n_samples * (self.n_times - 1)
    def __getitem__(self, idx):
        i, t = divmod(idx, self.n_times - 1)
        u_in  = (self.u[i, t]   - self.um) / self.us
        v_in  = (self.v[i, t]   - self.vm) / self.vs
        u_out = (self.u[i, t+1] - self.um) / self.us
        v_out = (self.v[i, t+1] - self.vm) / self.vs
        p_norm = (self.p[i] - self.pm) / self.ps
        return (torch.from_numpy(np.stack([u_in, v_in], 0)).float(),
                torch.from_numpy(np.stack([u_out, v_out], 0)).float(),
                torch.from_numpy(p_norm).float(),
                i)

@torch.no_grad()
def comprehensive_metrics(model, val_loader, device, n_trajectories):
    """Returns a dict of per-field metrics for Table 1, plus per-trajectory rel-L2 lists for Table 3."""
    model.eval()
    rus, rvs = [], []                          # per-sample rel-L2
    se_u_sum = se_v_sum = 0.0                  # squared error sums (for MSE)
    ae_u_sum = ae_v_sum = 0.0                  # absolute error sums (for MAE)
    max_ae_u = max_ae_v = 0.0
    n_elems = 0
    # accumulators for R^2 and Pearson (joint over all samples)
    sum_u_pred = sum_u_true = sum_u_p2 = sum_u_t2 = sum_u_pt = 0.0
    sum_v_pred = sum_v_true = sum_v_p2 = sum_v_t2 = sum_v_pt = 0.0
    # per-trajectory rel-L2 (averaged over (n_times - 1) pairs)
    traj_rel_u = np.zeros(n_trajectories)
    traj_rel_v = np.zeros(n_trajectories)
    traj_count = np.zeros(n_trajectories)
    for x, y, p, traj_idx in val_loader:
        x, y, p = x.to(device), y.to(device), p.to(device)
        pred = model(x, p)
        diff = pred - y
        # per-sample rel-L2
        num_u = (diff[:, 0] ** 2).sum(dim=-1); den_u = (y[:, 0] ** 2).sum(dim=-1)
        num_v = (diff[:, 1] ** 2).sum(dim=-1); den_v = (y[:, 1] ** 2).sum(dim=-1)
        ru = torch.sqrt(num_u / (den_u + 1e-12)).cpu().numpy()
        rv = torch.sqrt(num_v / (den_v + 1e-12)).cpu().numpy()
        rus.append(ru); rvs.append(rv)
        # MSE/MAE/max-AE
        se_u_sum += (diff[:, 0] ** 2).sum().item()
        se_v_sum += (diff[:, 1] ** 2).sum().item()
        ae_u_sum += diff[:, 0].abs().sum().item()
        ae_v_sum += diff[:, 1].abs().sum().item()
        max_ae_u = max(max_ae_u, diff[:, 0].abs().max().item())
        max_ae_v = max(max_ae_v, diff[:, 1].abs().max().item())
        n_elems += diff[:, 0].numel()
        # R^2 / Pearson accumulators
        u_p = pred[:, 0]; u_t = y[:, 0]
        v_p = pred[:, 1]; v_t = y[:, 1]
        sum_u_pred += u_p.sum().item(); sum_u_true += u_t.sum().item()
        sum_u_p2 += (u_p ** 2).sum().item(); sum_u_t2 += (u_t ** 2).sum().item()
        sum_u_pt += (u_p * u_t).sum().item()
        sum_v_pred += v_p.sum().item(); sum_v_true += v_t.sum().item()
        sum_v_p2 += (v_p ** 2).sum().item(); sum_v_t2 += (v_t ** 2).sum().item()
        sum_v_pt += (v_p * v_t).sum().item()
        # per-trajectory aggregates
        for j, ti in enumerate(traj_idx.numpy()):
            traj_rel_u[ti] += ru[j]
            traj_rel_v[ti] += rv[j]
            traj_count[ti] += 1
    rus = np.concatenate(rus)
    rvs = np.concatenate(rvs)
    N = n_elems
    mse_u = se_u_sum / N; mse_v = se_v_sum / N
    mae_u = ae_u_sum / N; mae_v = ae_v_sum / N
    # R^2 = 1 - SS_res / SS_tot
    mean_u_true = sum_u_true / N
    mean_v_true = sum_v_true / N
    ss_tot_u = sum_u_t2 - N * mean_u_true ** 2
    ss_tot_v = sum_v_t2 - N * mean_v_true ** 2
    r2_u = 1.0 - se_u_sum / max(ss_tot_u, 1e-12)
    r2_v = 1.0 - se_v_sum / max(ss_tot_v, 1e-12)
    # Pearson correlation
    mean_u_pred = sum_u_pred / N
    mean_v_pred = sum_v_pred / N
    cov_u = sum_u_pt / N - mean_u_pred * mean_u_true
    var_u_pred = sum_u_p2 / N - mean_u_pred ** 2
    var_u_true = sum_u_t2 / N - mean_u_true ** 2
    pearson_u = cov_u / max(np.sqrt(var_u_pred * var_u_true), 1e-12)
    cov_v = sum_v_pt / N - mean_v_pred * mean_v_true
    var_v_pred = sum_v_p2 / N - mean_v_pred ** 2
    var_v_true = sum_v_t2 / N - mean_v_true ** 2
    pearson_v = cov_v / max(np.sqrt(var_v_pred * var_v_true), 1e-12)
    # per-trajectory mean rel-L2
    traj_rel_u = traj_rel_u / np.maximum(traj_count, 1)
    traj_rel_v = traj_rel_v / np.maximum(traj_count, 1)
    return {
        'rel_u_mean': float(rus.mean()), 'rel_u_std': float(rus.std()),
        'rel_v_mean': float(rvs.mean()), 'rel_v_std': float(rvs.std()),
        'mse_u': mse_u, 'mse_v': mse_v,
        'mae_u': mae_u, 'mae_v': mae_v,
        'max_ae_u': max_ae_u, 'max_ae_v': max_ae_v,
        'r2_u': r2_u, 'r2_v': r2_v,
        'pearson_u': pearson_u, 'pearson_v': pearson_v,
        'traj_rel_u': traj_rel_u.tolist(),
        'traj_rel_v': traj_rel_v.tolist(),
    }
print('eval defined')

## Training loop

In [ ]:
def train_one(model_cls, seed, train_ds, val_ds, epochs, device, lr=4e-3, bs=256, log_path=None, model_kwargs=None):
    """Train one model and seed. Returns (state_dict, metrics, history)."""
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model_kwargs = model_kwargs or {}
    model = model_cls(**model_kwargs).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    tl = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
    vl = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.MSELoss()
    best_state = None
    best_loss = float('inf')
    history = []
    with open(log_path, 'w') as flog:
        flog.write(f'# {model_cls.__name__} seed={seed} params={n_params}\n')
        for ep in range(epochs):
            model.train()
            for x, y, p, _ in tl:
                x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True); p = p.to(device, non_blocking=True)
                opt.zero_grad()
                pred = model(x, p)
                loss = crit(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            # quick val pass, rel-L2 only
            model.eval()
            ru_n = ru_d = rv_n = rv_d = 0.0
            with torch.no_grad():
                for x, y, p, _ in vl:
                    x = x.to(device); y = y.to(device); p = p.to(device)
                    pred = model(x, p)
                    ru_n += ((pred[:,0]-y[:,0])**2).sum().item()
                    ru_d += (y[:,0]**2).sum().item()
                    rv_n += ((pred[:,1]-y[:,1])**2).sum().item()
                    rv_d += (y[:,1]**2).sum().item()
            rel_u_glob = (ru_n / max(ru_d, 1e-12)) ** 0.5
            rel_v_glob = (rv_n / max(rv_d, 1e-12)) ** 0.5
            sch.step()
            flog.write(f'epoch {ep} rel_u {rel_u_glob:.6f} rel_v {rel_v_glob:.6f}\n'); flog.flush()
            score = rel_u_glob + rel_v_glob
            if score < best_loss:
                best_loss = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            history.append({'epoch': ep, 'rel_u_glob': rel_u_glob, 'rel_v_glob': rel_v_glob})
    # load best, run full eval
    model.load_state_dict(best_state)
    metrics = comprehensive_metrics(model, vl, device, n_trajectories=val_ds.n_samples)
    metrics['n_params'] = n_params
    metrics['model_cls'] = model_cls.__name__
    metrics['seed'] = seed
    return best_state, metrics, history
print('train_one defined')

## Build datasets (80/20 split)

In [ ]:
cfg = DataConfig()
with h5py.File(LOCAL_DATA, 'r') as f:
    N = f['u_traj'].shape[0]; nx = f['u_traj'].shape[-1]
print(f'Dataset: {N} trajectories, nx={nx}')
rng = np.random.RandomState(42)
perm = rng.permutation(N)
n_train = int(N * 0.8)
train_idx = np.sort(perm[:n_train]); val_idx = np.sort(perm[n_train:])
with h5py.File(LOCAL_DATA, 'r') as f:
    u = np.array(f['u_traj'][train_idx]); v = np.array(f['v_traj'][train_idx])
um, us = float(u.mean()), float(u.std() + 1e-8)
vm, vs = float(v.mean()), float(v.std() + 1e-8)
del u, v
ranges = np.array([cfg.Du_range, cfg.Dv_range, cfg.a_range, cfg.b_range, cfg.tau_range])
pm = ranges.mean(axis=1).astype(np.float32)
ps = ((ranges[:, 1] - ranges[:, 0]) / np.sqrt(12.0)).astype(np.float32)
print(f'u: mean={um:.4f} std={us:.4f}; v: mean={vm:.4f} std={vs:.4f}')
train_ds = H5SingleStepDataset(LOCAL_DATA, train_idx, um, us, vm, vs, pm, ps)
val_ds   = H5SingleStepDataset(LOCAL_DATA, val_idx,   um, us, vm, vs, pm, ps)
print(f'train: {len(train_ds.u)} traj, val: {len(val_ds.u)} traj')
device = 'cuda'
SEEDS = [42, 1337, 2025, 314, 271]

## Tables 1 and 2: Parametric FNO

Headline model. Resumable: checks Drive for existing checkpoints and metrics.

In [ ]:
PARAM_METRICS_PATH = os.path.join(DRIVE_FOLDER, 'parametric_fno_metrics.json')
param_runs = json.load(open(PARAM_METRICS_PATH)) if os.path.exists(PARAM_METRICS_PATH) else []
done_seeds = {r['seed'] for r in param_runs}
for seed in SEEDS:
    if seed in done_seeds:
        print(f'== skip Parametric FNO seed={seed} (done) =='); continue
    print(f'\n== Parametric FNO, seed={seed} ==', flush=True)
    t0 = time.time()
    log_path = os.path.join(DRIVE_FOLDER, f'run_parametric_s{seed}.log')
    state, m, hist = train_one(ParametricFNO, seed, train_ds, val_ds, epochs=40, device=device,
                                log_path=log_path)
    m['wall_train_s'] = time.time() - t0
    ck_path = os.path.join(DRIVE_FOLDER, 'checkpoints', f'parametric_s{seed}.pt')
    torch.save({'state_dict': state, 'um': um, 'us': us, 'vm': vm, 'vs': vs, 'pm': pm.tolist(), 'ps': ps.tolist(),
                'model_cls': 'ParametricFNO'}, ck_path)
    # drop per-traj lists from the JSON to save space
    m_summary = {k: v for k, v in m.items() if k not in ('traj_rel_u', 'traj_rel_v')}
    param_runs.append(m_summary)
    # keep per-traj lists separately for Table 3
    np.savez(os.path.join(DRIVE_FOLDER, f'traj_rel_parametric_s{seed}.npz'),
             rel_u=m['traj_rel_u'], rel_v=m['traj_rel_v'])
    with open(PARAM_METRICS_PATH, 'w') as f: json.dump(param_runs, f, indent=2)
    print(f'  rel_u={m["rel_u_mean"]:.5f} +/- {m["rel_u_std"]:.5f}  rel_v={m["rel_v_mean"]:.5f} +/- {m["rel_v_std"]:.5f}  wall={m["wall_train_s"]:.0f}s')
print('\nParametric FNO done.')

## Table 2: Broadcast FNO baseline

In [ ]:
BCAST_METRICS_PATH = os.path.join(DRIVE_FOLDER, 'broadcast_fno_metrics.json')
bcast_runs = json.load(open(BCAST_METRICS_PATH)) if os.path.exists(BCAST_METRICS_PATH) else []
done_seeds = {r['seed'] for r in bcast_runs}
for seed in SEEDS:
    if seed in done_seeds:
        print(f'== skip Broadcast FNO seed={seed} (done) =='); continue
    print(f'\n== Broadcast FNO, seed={seed} ==', flush=True)
    t0 = time.time()
    log_path = os.path.join(DRIVE_FOLDER, f'run_broadcast_s{seed}.log')
    state, m, hist = train_one(BroadcastFNO, seed, train_ds, val_ds, epochs=40, device=device,
                                log_path=log_path)
    m['wall_train_s'] = time.time() - t0
    ck_path = os.path.join(DRIVE_FOLDER, 'checkpoints', f'broadcast_s{seed}.pt')
    torch.save({'state_dict': state, 'um': um, 'us': us, 'vm': vm, 'vs': vs, 'pm': pm.tolist(), 'ps': ps.tolist(),
                'model_cls': 'BroadcastFNO'}, ck_path)
    m_summary = {k: v for k, v in m.items() if k not in ('traj_rel_u', 'traj_rel_v')}
    bcast_runs.append(m_summary)
    with open(BCAST_METRICS_PATH, 'w') as f: json.dump(bcast_runs, f, indent=2)
    print(f'  rel_u={m["rel_u_mean"]:.5f} +/- {m["rel_u_std"]:.5f}  rel_v={m["rel_v_mean"]:.5f} +/- {m["rel_v_std"]:.5f}  wall={m["wall_train_s"]:.0f}s')
print('\nBroadcast FNO done.')

## Table 2: DeepONet baseline

In [ ]:
DEEPONET_METRICS_PATH = os.path.join(DRIVE_FOLDER, 'deeponet_metrics.json')
don_runs = json.load(open(DEEPONET_METRICS_PATH)) if os.path.exists(DEEPONET_METRICS_PATH) else []
done_seeds = {r['seed'] for r in don_runs}
for seed in SEEDS:
    if seed in done_seeds:
        print(f'== skip DeepONet seed={seed} (done) =='); continue
    print(f'\n== DeepONet, seed={seed} ==', flush=True)
    t0 = time.time()
    log_path = os.path.join(DRIVE_FOLDER, f'run_deeponet_s{seed}.log')
    state, m, hist = train_one(DeepONet, seed, train_ds, val_ds, epochs=40, device=device,
                                log_path=log_path, model_kwargs={'nx': nx})
    m['wall_train_s'] = time.time() - t0
    ck_path = os.path.join(DRIVE_FOLDER, 'checkpoints', f'deeponet_s{seed}.pt')
    torch.save({'state_dict': state, 'um': um, 'us': us, 'vm': vm, 'vs': vs, 'pm': pm.tolist(), 'ps': ps.tolist(),
                'model_cls': 'DeepONet'}, ck_path)
    m_summary = {k: v for k, v in m.items() if k not in ('traj_rel_u', 'traj_rel_v')}
    don_runs.append(m_summary)
    with open(DEEPONET_METRICS_PATH, 'w') as f: json.dump(don_runs, f, indent=2)
    print(f'  rel_u={m["rel_u_mean"]:.5f} +/- {m["rel_u_std"]:.5f}  rel_v={m["rel_v_mean"]:.5f} +/- {m["rel_v_std"]:.5f}  wall={m["wall_train_s"]:.0f}s')
print('\nDeepONet done.')

## Table 3: parameter-error correlation

Pearson correlation between each parameter and per-trajectory rel-L2, using the Parametric FNO (seed 42) results from above.

In [ ]:
from scipy import stats
# per-trajectory rel-L2 from the seed-42 run
traj_path = os.path.join(DRIVE_FOLDER, 'traj_rel_parametric_s42.npz')
if not os.path.exists(traj_path):
    raise FileNotFoundError(f'{traj_path} -- run Cell 8 first')
tr = np.load(traj_path)
traj_rel_u, traj_rel_v = tr['rel_u'], tr['rel_v']
with h5py.File(LOCAL_DATA, 'r') as f:
    val_params = np.array(f['params'][val_idx])  # (n_val, 5)
print(f'val_params shape={val_params.shape}, traj_rel_u shape={traj_rel_u.shape}')
param_names = ['Du', 'Dv', 'a', 'b', 'tau']
rho_u = {}; rho_v = {}
for i, pn in enumerate(param_names):
    ru, pu = stats.pearsonr(val_params[:, i], traj_rel_u)
    rv, pv = stats.pearsonr(val_params[:, i], traj_rel_v)
    rho_u[pn] = {'rho': float(ru), 'p': float(pu)}
    rho_v[pn] = {'rho': float(rv), 'p': float(pv)}
    print(f'{pn:>5s}: rho_u={ru:+.3f}, rho_v={rv:+.3f}')
with open(os.path.join(DRIVE_FOLDER, 'table3_param_correlation.json'), 'w') as f:
    json.dump({'rho_u': rho_u, 'rho_v': rho_v, 'source': 'Parametric FNO seed 42'}, f, indent=2)
print('Table 3 saved.')

## Table 4: computational efficiency

Times the Parametric FNO forward pass (batch 1 and 32) against the semi-implicit FD solver on A100.

In [ ]:
# best Parametric FNO checkpoint (seed 42)
ck = torch.load(os.path.join(DRIVE_FOLDER, 'checkpoints', 'parametric_s42.pt'), map_location=device, weights_only=False)
model = ParametricFNO().to(device)
model.load_state_dict(ck['state_dict']); model.eval()

def time_fno(model, bs, n_steps=50, repeats=50):
    x = torch.randn(bs, 2, nx, device=device)
    p = torch.zeros(bs, 5, device=device)
    with torch.no_grad():
        for _ in range(10): _ = model(x, p)  # warmup
    torch.cuda.synchronize()
    times = []
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        for _ in range(repeats):
            t0 = time.time()
            cur = x.clone()
            for _ in range(n_steps):
                cur = model(cur, p)
            torch.cuda.synchronize()
            times.append(time.time() - t0)
    mem_mb = torch.cuda.max_memory_allocated() / 1e6
    times = np.array(times)
    return {'mean_total_s': float(times.mean()), 'std_total_s': float(times.std()),
            'per_step_ms': float(times.mean() / n_steps * 1000),
            'per_sample_per_step_ms': float(times.mean() / n_steps / bs * 1000),
            'mem_mb': float(mem_mb)}

fno_b1  = time_fno(model, bs=1)
fno_b32 = time_fno(model, bs=32)
print('FNO bs=1: ', fno_b1)
print('FNO bs=32:', fno_b32)

# FD solver benchmark, semi-implicit as in the repo
def fd_solve_batch_torch(u0, v0, params, T, dt, n_save, nx, device='cuda'):
    """Torch-batched semi-implicit FD on GPU. params: (B, 5)."""
    B = u0.shape[0]
    dx = 1.0 / nx
    lap = torch.zeros(nx, nx, device=device, dtype=torch.float32)
    c = 1.0 / (dx * dx)
    for i in range(nx):
        lap[i, i] = -2 * c
        lap[i, (i + 1) % nx] = c
        lap[i, (i - 1) % nx] = c
    Du = params[:, 0].view(B, 1, 1)
    Dv = params[:, 1].view(B, 1, 1)
    a  = params[:, 2].view(B, 1)
    b  = params[:, 3].view(B, 1)
    tau = params[:, 4].view(B, 1)
    I = torch.eye(nx, device=device).unsqueeze(0).expand(B, nx, nx)
    A_u = I - dt * Du * lap.unsqueeze(0).expand(B, nx, nx)
    A_v = I - dt * Dv * lap.unsqueeze(0).expand(B, nx, nx)
    LU_u, piv_u = torch.linalg.lu_factor(A_u)
    LU_v, piv_v = torch.linalg.lu_factor(A_v)
    u, v = u0.clone(), v0.clone()
    n_steps = int(T / dt)
    for _ in range(n_steps):
        reaction_u = u - u**3/3 - v
        reaction_v = (u + a - b * v) / tau
        rhs_u = u + dt * reaction_u
        rhs_v = v + dt * reaction_v
        u = torch.linalg.lu_solve(LU_u, piv_u, rhs_u.unsqueeze(-1)).squeeze(-1)
        v = torch.linalg.lu_solve(LU_v, piv_v, rhs_v.unsqueeze(-1)).squeeze(-1)
    return u, v

def time_fd(bs, T=0.5, dt=0.01, n_save=50, repeats=10):
    u0 = torch.randn(bs, nx, device=device)
    v0 = torch.randn(bs, nx, device=device)
    params = torch.rand(bs, 5, device=device)
    params[:, 0] = 0.01 + params[:, 0] * 0.09  # Du
    params[:, 1] = 0.005 + params[:, 1] * 0.045  # Dv
    params[:, 2] = -0.1 + params[:, 2] * 0.2  # a
    params[:, 3] = 0.1 + params[:, 3] * 0.4   # b
    params[:, 4] = 1.0 + params[:, 4] * 19.0  # tau
    for _ in range(2): _ = fd_solve_batch_torch(u0, v0, params, T, dt, n_save, nx, device=device)
    torch.cuda.synchronize()
    times = []
    torch.cuda.reset_peak_memory_stats()
    for _ in range(repeats):
        t0 = time.time()
        _ = fd_solve_batch_torch(u0, v0, params, T, dt, n_save, nx, device=device)
        torch.cuda.synchronize()
        times.append(time.time() - t0)
    mem_mb = torch.cuda.max_memory_allocated() / 1e6
    return {'mean_total_s': float(np.mean(times)), 'per_step_ms': float(np.mean(times) / int(T/dt) * 1000),
            'mem_mb': float(mem_mb)}

fd_b32 = time_fd(bs=32)
print('FD bs=32: ', fd_b32)
speedup_b1 = fd_b32['per_step_ms'] / fno_b1['per_step_ms']
speedup_b32 = fd_b32['per_step_ms'] / fno_b32['per_step_ms'] * 32  # FNO bs=32 amortises
print(f'Speedup FNO(bs=1) vs FD: {speedup_b1:.1f}x')
print(f'Speedup FNO(bs=32) per-sample vs FD: {speedup_b32:.1f}x')
with open(os.path.join(DRIVE_FOLDER, 'table4_efficiency.json'), 'w') as f:
    json.dump({'fno_bs1': fno_b1, 'fno_bs32': fno_b32, 'fd_bs32': fd_b32,
               'speedup_bs1': speedup_b1, 'speedup_bs32': speedup_b32}, f, indent=2)
print('Table 4 saved.')

## Table 5: out-of-range extrapolation

Generate 100 test trajectories per out-of-range scenario and evaluate the Parametric FNO (seed 42).

In [ ]:
# reuse the GPU FD solver to generate test trajectories
def fd_traj(u0, v0, params, T, dt, n_save):
    """Generate n_save+1 trajectory frames. params: (B, 5)."""
    B = u0.shape[0]
    dx = 1.0 / nx
    lap = torch.zeros(nx, nx, device=device, dtype=torch.float32)
    c = 1.0 / (dx * dx)
    for i in range(nx):
        lap[i, i] = -2 * c
        lap[i, (i + 1) % nx] = c
        lap[i, (i - 1) % nx] = c
    Du = params[:, 0].view(B, 1, 1); Dv = params[:, 1].view(B, 1, 1)
    a  = params[:, 2].view(B, 1);    b  = params[:, 3].view(B, 1); tau = params[:, 4].view(B, 1)
    I = torch.eye(nx, device=device).unsqueeze(0).expand(B, nx, nx)
    A_u = I - dt * Du * lap.unsqueeze(0).expand(B, nx, nx)
    A_v = I - dt * Dv * lap.unsqueeze(0).expand(B, nx, nx)
    LU_u, piv_u = torch.linalg.lu_factor(A_u)
    LU_v, piv_v = torch.linalg.lu_factor(A_v)
    n_steps = int(T / dt)
    save_interval = max(1, n_steps // n_save)
    u, v = u0.clone(), v0.clone()
    u_hist = [u.clone()]; v_hist = [v.clone()]
    for step in range(n_steps):
        reaction_u = u - u**3/3 - v
        reaction_v = (u + a - b * v) / tau
        rhs_u = u + dt * reaction_u
        rhs_v = v + dt * reaction_v
        u = torch.linalg.lu_solve(LU_u, piv_u, rhs_u.unsqueeze(-1)).squeeze(-1)
        v = torch.linalg.lu_solve(LU_v, piv_v, rhs_v.unsqueeze(-1)).squeeze(-1)
        if (step + 1) % save_interval == 0:
            u_hist.append(u.clone()); v_hist.append(v.clone())
    return torch.stack(u_hist, dim=1), torch.stack(v_hist, dim=1)

def sample_grf_ic(B, nx, alpha=2.0, device='cuda', seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    k = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    k_abs = k.abs(); k_abs[0] = 1e-10
    power = (1 + k_abs**2) ** (-alpha/2)
    def make(B, scale):
        u_hat = (torch.randn(B, nx, generator=g, device=device) + 1j * torch.randn(B, nx, generator=g, device=device)) * power.unsqueeze(0) * scale
        u = torch.fft.ifft(u_hat).real
        u = (u - u.mean(dim=-1, keepdim=True)) / (u.std(dim=-1, keepdim=True) + 1e-8)
        return u
    return make(B, 1.0), make(B, 0.5)

T, dt, n_save = 1.0, 0.01, 50
scenarios = {
    'low_diff':   {'Du': 0.005, 'Dv': 0.002, 'a': 0.0, 'b': 0.3, 'tau': 10.0},
    'high_diff':  {'Du': 0.15,  'Dv': 0.08,  'a': 0.0, 'b': 0.3, 'tau': 10.0},
    'high_tau':   {'Du': 0.05,  'Dv': 0.025, 'a': 0.0, 'b': 0.3, 'tau': 30.0},
    'low_recovery': {'Du': 0.05, 'Dv': 0.025, 'a': 0.0, 'b': 0.05, 'tau': 10.0},
}
n_test = 100
table5 = {}
for name, p in scenarios.items():
    print(f'\n-- scenario {name}: {p} --')
    params = torch.tensor([[p['Du'], p['Dv'], p['a'], p['b'], p['tau']]] * n_test, device=device)
    u0, v0 = sample_grf_ic(n_test, nx, seed=hash(name) & 0xffff)
    # FD ground truth
    u_true, v_true = fd_traj(u0, v0, params, T, dt, n_save)  # (B, n_save+1, nx)
    # FNO single-step pairs
    rel_u_list, rel_v_list = [], []
    with torch.no_grad():
        # normalise inputs
        for t in range(n_save):
            x = torch.stack([(u_true[:, t] - um) / us, (v_true[:, t] - vm) / vs], dim=1)
            p_norm = (params - torch.tensor(pm, device=device)) / torch.tensor(ps, device=device)
            pred = model(x, p_norm)
            u_pred = pred[:, 0] * us + um
            v_pred = pred[:, 1] * vs + vm
            num_u = ((u_pred - u_true[:, t+1])**2).sum(dim=-1); den_u = (u_true[:, t+1]**2).sum(dim=-1)
            num_v = ((v_pred - v_true[:, t+1])**2).sum(dim=-1); den_v = (v_true[:, t+1]**2).sum(dim=-1)
            rel_u_list.append(torch.sqrt(num_u/(den_u+1e-12)).cpu().numpy())
            rel_v_list.append(torch.sqrt(num_v/(den_v+1e-12)).cpu().numpy())
    rel_u_arr = np.array(rel_u_list).flatten()  # over (n_save, n_test)
    rel_v_arr = np.array(rel_v_list).flatten()
    table5[name] = {'rel_u_mean': float(rel_u_arr.mean()), 'rel_u_std': float(rel_u_arr.std()),
                    'rel_v_mean': float(rel_v_arr.mean()), 'rel_v_std': float(rel_v_arr.std()),
                    'params': p}
    print(f'  rel_u={rel_u_arr.mean():.4f} +/- {rel_u_arr.std():.4f}, rel_v={rel_v_arr.mean():.4f} +/- {rel_v_arr.std():.4f}')
with open(os.path.join(DRIVE_FOLDER, 'table5_extrapolation.json'), 'w') as f:
    json.dump(table5, f, indent=2)
print('Table 5 saved.')

## Generate LaTeX for all tables

In [ ]:
import json, numpy as np
def fmt(x, sig=4):
    return f'{x:.{sig}g}'
def agg(runs, key):
    vals = np.array([r[key] for r in runs])
    return vals.mean(), vals.std()

param_runs = json.load(open(os.path.join(DRIVE_FOLDER, 'parametric_fno_metrics.json')))
bcast_runs = json.load(open(os.path.join(DRIVE_FOLDER, 'broadcast_fno_metrics.json')))
don_runs   = json.load(open(os.path.join(DRIVE_FOLDER, 'deeponet_metrics.json')))
table3 = json.load(open(os.path.join(DRIVE_FOLDER, 'table3_param_correlation.json')))
table4 = json.load(open(os.path.join(DRIVE_FOLDER, 'table4_efficiency.json')))
table5 = json.load(open(os.path.join(DRIVE_FOLDER, 'table5_extrapolation.json')))

tables_tex = []

# Table 1: single-step metrics
metrics_order = [('rel_L2', 'rel_u_mean', 'rel_u_std', 'rel_v_mean', 'rel_v_std'),
                 ('MSE',    'mse_u',      None,        'mse_v',      None),
                 ('MAE',    'mae_u',      None,        'mae_v',      None),
                 ('Max AE', 'max_ae_u',   None,        'max_ae_v',   None),
                 ('R^2',    'r2_u',       None,        'r2_v',       None),
                 ('Pearson','pearson_u',  None,        'pearson_v',  None)]
lines1 = []
lines1.append(r'\begin{table*}[t]')
lines1.append(r'\caption{Single-step prediction accuracy on the held-out validation set ($N_{\text{test}}=1{,}600$ trajectories, $n_x=256$). Metrics computed in normalised space; rel-$L^2$ reported as per-sample mean across val samples ($\pm$ std across 5 training seeds), other metrics as element-wise averages.}')
lines1.append(r'\label{tab:single_step_metrics}')
lines1.append(r'\centering')
lines1.append(r'\begin{small}\begin{sc}')
lines1.append(r'\begin{tabular}{lcc}')
lines1.append(r'\toprule')
lines1.append(r'Metric & Activator ($u$) & Inhibitor ($v$) \\')
lines1.append(r'\midrule')
for label, k_um, k_us, k_vm, k_vs in metrics_order:
    um_, um_s = agg(param_runs, k_um); vm_, vm_s = agg(param_runs, k_vm)
    cell_u = f'${um_:.4g} \\pm {um_s:.2g}$'
    cell_v = f'${vm_:.4g} \\pm {vm_s:.2g}$'
    lines1.append(f'{label} & {cell_u} & {cell_v} \\\\')
lines1.append(r'\bottomrule')
lines1.append(r'\end{tabular}\end{sc}\end{small}\end{table*}')
tables_tex.append('\n'.join(lines1))

# Table 2: baselines
def row(runs, label, with_params=True):
    rum, rus = agg(runs, 'rel_u_mean'); rvm, rvs = agg(runs, 'rel_v_mean')
    mam, mas = agg(runs, 'mae_u');     vam, vas = agg(runs, 'mae_v')
    np_ = runs[0]['n_params'] / 1e6
    cell = lambda m, s: f'${m:.4g} \\pm {s:.2g}$'
    extra = f' & ${np_:.2f}$M' if with_params else ''
    return f'{label} & {cell(rum,rus)} & {cell(rvm,rvs)} & {cell(mam,mas)} & {cell(vam,vas)}{extra} \\\\'
lines2 = []
lines2.append(r'\begin{table*}[t]')
lines2.append(r'\caption{Baseline comparison on the validation set, mean $\pm$ std across $n_{\text{seeds}}=5$ independent training runs.}')
lines2.append(r'\label{tab:baselines}')
lines2.append(r'\centering\begin{small}\begin{sc}\setlength{\tabcolsep}{6pt}')
lines2.append(r'\begin{tabular}{lccccc}')
lines2.append(r'\toprule')
lines2.append(r' & \multicolumn{2}{c}{$\epsilon_{\text{rel}}$} & \multicolumn{2}{c}{MAE} & \\')
lines2.append(r'\cmidrule(lr){2-3} \cmidrule(lr){4-5}')
lines2.append(r'Method & $u$ & $v$ & $u$ & $v$ & Params \\')
lines2.append(r'\midrule')
lines2.append(row(don_runs,   'DeepONet'))
lines2.append(row(bcast_runs, 'Baseline FNO'))
lines2.append(row(param_runs, 'Parametric FNO'))
lines2.append(r'\bottomrule')
lines2.append(r'\end{tabular}\end{sc}\end{small}\end{table*}')
tables_tex.append('\n'.join(lines2))

# Table 3: parameter correlation
lines3 = []
lines3.append(r'\begin{table}[t]')
lines3.append(r'\caption{Pearson correlation between each FHN parameter and per-trajectory single-step rel-$L^2$ on the validation set (Parametric FNO, seed 42).}')
lines3.append(r'\label{tab:parameter_correlation}')
lines3.append(r'\centering\begin{small}\begin{sc}')
lines3.append(r'\begin{tabular}{lccccc}')
lines3.append(r'\toprule')
lines3.append(r' & $D_u$ & $D_v$ & $a$ & $b$ & $\tau$ \\')
lines3.append(r'\midrule')
def fmt_rho(x): return f'${x:+.3f}$'
lines3.append('$u$ error & ' + ' & '.join(fmt_rho(table3['rho_u'][p]['rho']) for p in ['Du','Dv','a','b','tau']) + r' \\')
lines3.append('$v$ error & ' + ' & '.join(fmt_rho(table3['rho_v'][p]['rho']) for p in ['Du','Dv','a','b','tau']) + r' \\')
lines3.append(r'\bottomrule')
lines3.append(r'\end{tabular}\end{sc}\end{small}\end{table}')
tables_tex.append('\n'.join(lines3))

# Table 4: efficiency
lines4 = []
lines4.append(r'\begin{table}[t]')
lines4.append(r'\caption{Computational efficiency: FNO vs. semi-implicit FD solver, $n_x=256$, $n_{\text{steps}}=50$, NVIDIA A100.}')
lines4.append(r'\label{tab:efficiency_comparison}')
lines4.append(r'\centering\begin{small}\setlength{\tabcolsep}{4pt}')
lines4.append(r'\begin{tabular}{lccc}')
lines4.append(r'\toprule')
lines4.append(r'Method & Time/step & Total (50 steps) & Memory \\')
lines4.append(r'\midrule')
fno1 = table4['fno_bs1']; fno32 = table4['fno_bs32']; fd = table4['fd_bs32']
lines4.append(f'FD (batch 32) & ${fd["per_step_ms"]:.3g}$ ms & ${fd["mean_total_s"]:.3g}$ s & ${fd["mem_mb"]:.0f}$ MB \\\\')
lines4.append(f'FNO (batch 1) & ${fno1["per_step_ms"]:.3g}$ ms & ${fno1["mean_total_s"]:.3g}$ s & ${fno1["mem_mb"]:.0f}$ MB \\\\')
lines4.append(f'FNO (batch 32) & ${fno32["per_sample_per_step_ms"]:.3g}$ ms & ${fno32["mean_total_s"]:.3g}$ s & ${fno32["mem_mb"]:.0f}$ MB \\\\')
lines4.append(r'\midrule')
lines4.append(f'Speedup & ${table4["speedup_bs1"]:.0f}\\times$ / ${table4["speedup_bs32"]:.0f}\\times$ & --- & --- \\\\')
lines4.append(r'\bottomrule')
lines4.append(r'\end{tabular}\end{small}\end{table}')
tables_tex.append('\n'.join(lines4))

# Table 5: extrapolation
labels = {'low_diff': 'Low diffusion ($D_u{=}0.005$, $D_v{=}0.002$)',
          'high_diff': 'High diffusion ($D_u{=}0.15$, $D_v{=}0.08$)',
          'high_tau': 'High time-scale separation ($\\tau{=}30$)',
          'low_recovery': 'Low recovery ($b{=}0.05$)'}
lines5 = []
lines5.append(r'\begin{table}[t]')
lines5.append(r'\caption{Extrapolation performance: relative $L^2$ error (per-sample mean) on 100 test trajectories per out-of-range scenario, evaluated with the Parametric FNO (seed 42).}')
lines5.append(r'\label{tab:extrapolation}')
lines5.append(r'\centering\begin{sc}\fontsize{8}{9.6}\selectfont\setlength{\tabcolsep}{4pt}')
lines5.append(r'\begin{tabular}{lcc}')
lines5.append(r'\toprule')
lines5.append(r'Regime & $\epsilon_{\text{rel}}(u)$ & $\epsilon_{\text{rel}}(v)$ \\')
lines5.append(r'\midrule')
# within-range row: Parametric FNO from Table 1
rum, _ = agg(param_runs, 'rel_u_mean'); rvm, _ = agg(param_runs, 'rel_v_mean')
lines5.append(f'Within range (validation) & ${rum:.4g}$ & ${rvm:.4g}$ \\\\')
for k, label in labels.items():
    e = table5[k]
    lines5.append(f'{label} & ${e["rel_u_mean"]:.4g}$ & ${e["rel_v_mean"]:.4g}$ \\\\')
lines5.append(r'\bottomrule')
lines5.append(r'\end{tabular}\end{sc}\end{table}')
tables_tex.append('\n'.join(lines5))

tex_out = '\n\n'.join(tables_tex)
with open(os.path.join(DRIVE_FOLDER, 'tables.tex'), 'w') as f:
    f.write(tex_out)
print(tex_out)
print('\nAll tables saved to', os.path.join(DRIVE_FOLDER, 'tables.tex'))

## Regenerate model-dependent figures

Rebuilds single_step_regimes.png, phase_portraits.png, and rollout_error_evolution.png from parametric_s42.pt. Inference only, no retraining.

In [ ]:
# Regenerate figures from parametric_s42.pt. Inference only. Run after Cell 16.
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np, h5py, torch, os

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.labelsize': 11,
    'axes.titlesize': 12, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.titlesize': 13,
})
COL = {'true': '#1f77b4', 'pred': '#ff7f0e', 'error': '#d62728', 'start': '#2ca02c'}
FIG_DIR = os.path.join(DRIVE_FOLDER, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)

# load the checkpoint and its normalization stats
ck = torch.load(os.path.join(DRIVE_FOLDER, 'checkpoints', 'parametric_s42.pt'),
                map_location=device, weights_only=False)
fig_model = ParametricFNO().to(device)
fig_model.load_state_dict(ck['state_dict']); fig_model.eval()
_um, _us, _vm, _vs = ck['um'], ck['us'], ck['vm'], ck['vs']
_pm = np.asarray(ck['pm'], dtype=np.float32); _ps = np.asarray(ck['ps'], dtype=np.float32)
print(f'loaded parametric_s42.pt  (um={_um:.3f} us={_us:.3f} vm={_vm:.3f} vs={_vs:.3f})')

@torch.no_grad()
def _single_step(u_in, v_in, params):
    un = (u_in - _um) / _us; vn = (v_in - _vm) / _vs
    x = torch.tensor(np.stack([un, vn], 0), dtype=torch.float32).unsqueeze(0).to(device)
    p = torch.tensor((params - _pm) / _ps, dtype=torch.float32).unsqueeze(0).to(device)
    out = fig_model(x, p)[0].cpu().numpy()
    return out[0] * _us + _um, out[1] * _vs + _vm

@torch.no_grad()
def _rollout(u0, v0, params, n_steps):
    un = (u0 - _um) / _us; vn = (v0 - _vm) / _vs
    cur = torch.tensor(np.stack([un, vn], 0), dtype=torch.float32).unsqueeze(0).to(device)
    p = torch.tensor((params - _pm) / _ps, dtype=torch.float32).unsqueeze(0).to(device)
    frames = [cur.clone()]
    for _ in range(n_steps):
        cur = fig_model(cur, p); frames.append(cur.clone())
    arr = torch.cat(frames, 0).cpu().numpy()      # (n_steps+1, 2, nx)
    return arr[:, 0] * _us + _um, arr[:, 1] * _vs + _vm

def _relL2(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-12)

# held-out trajectories from the H5
with h5py.File(LOCAL_DATA, 'r') as f:
    P = np.array(f['params'][val_idx])
    U = np.array(f['u_traj'][val_idx])
    V = np.array(f['v_traj'][val_idx])
n_times = U.shape[1]; nx = U.shape[-1]
x = np.linspace(0, 1, nx)
N_STEPS = min(50, n_times - 1)
T_MID = n_times // 2
print(f'held-out: {U.shape[0]} traj, n_times={n_times}, nx={nx}, rollout={N_STEPS}')

# Figure A: single-step predictions across diffusion regimes
Du = P[:, 0]
regimes = [(int(np.argmin(Du)), 'Low Diffusion', 0),
           (int(np.argsort(Du)[len(Du) // 2]), 'Medium Diffusion', 1),
           (int(np.argmax(Du)), 'High Diffusion', 2)]
fig = plt.figure(figsize=(16, 12)); gs = gridspec.GridSpec(3, 3, hspace=0.3, wspace=0.3)
for s_idx, name, row in regimes:
    t = T_MID
    u_t, v_t = U[s_idx, t + 1], V[s_idx, t + 1]
    u_p, v_p = _single_step(U[s_idx, t], V[s_idx, t], P[s_idx])
    eu = np.abs(u_t - u_p); ev = np.abs(v_t - v_p)
    axu = plt.subplot(gs[row, 0])
    axu.plot(x, u_t, color=COL['true'], lw=2, label='True')
    axu.plot(x, u_p, '--', color=COL['pred'], lw=2, label='Predicted')
    axu.fill_between(x, u_t, u_p, alpha=0.3, color=COL['error'])
    axu.set_ylabel('u'); axu.set_title(f'{name}: u (MAE: {eu.mean():.3e})', fontweight='bold')
    axu.grid(True, alpha=0.3)
    if row == 0: axu.legend()
    Du_, Dv_, a_, b_, tau_ = P[s_idx]
    axu.text(0.02, 0.98, f'Du={Du_:.3f}, Dv={Dv_:.3f}\na={a_:.2f}, b={b_:.2f}, tau={tau_:.1f}',
             transform=axu.transAxes, va='top', fontsize=8,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axv = plt.subplot(gs[row, 1])
    axv.plot(x, v_t, color=COL['true'], lw=2, label='True')
    axv.plot(x, v_p, '--', color=COL['pred'], lw=2, label='Predicted')
    axv.fill_between(x, v_t, v_p, alpha=0.3, color=COL['error'])
    axv.set_ylabel('v'); axv.set_title(f'{name}: v (MAE: {ev.mean():.3e})', fontweight='bold')
    axv.grid(True, alpha=0.3)
    axe = plt.subplot(gs[row, 2])
    axe.plot(x, eu, color=COL['true'], lw=2, label='u error')
    axe.plot(x, ev, color=COL['pred'], lw=2, label='v error')
    axe.set_ylabel('Absolute Error'); axe.set_title(f'{name}: Errors', fontweight='bold')
    axe.grid(True, alpha=0.3); axe.legend()
p1 = os.path.join(FIG_DIR, 'single_step_regimes.png')
plt.savefig(p1, dpi=300, bbox_inches='tight'); plt.close(); print('saved', p1)

# Figure B: phase portraits, rollout vs ground truth
s_idx = 0
u_pred, v_pred = _rollout(U[s_idx, 0], V[s_idx, 0], P[s_idx], N_STEPS)
u_true = U[s_idx, :N_STEPS + 1]; v_true = V[s_idx, :N_STEPS + 1]
sp = [nx // 4, nx // 2, 3 * nx // 4]
fig, axes = plt.subplots(1, len(sp), figsize=(5 * len(sp), 5))
for i, xi in enumerate(sp):
    ax = axes[i]
    ax.plot(u_true[:, xi], v_true[:, xi], color=COL['true'], lw=2.5, label='Ground Truth', alpha=0.8)
    ax.plot(u_pred[:, xi], v_pred[:, xi], '--', color=COL['pred'], lw=2, label='Prediction', alpha=0.8)
    ax.scatter(u_true[0, xi], v_true[0, xi], c=COL['start'], s=150, marker='o', label='Start',
               zorder=10, edgecolors='black', linewidths=1.5)
    ax.scatter(u_true[-1, xi], v_true[-1, xi], c=COL['true'], s=150, marker='s', label='True End',
               zorder=10, edgecolors='black', linewidths=1.5)
    ax.scatter(u_pred[-1, xi], v_pred[-1, xi], c=COL['pred'], s=150, marker='^', label='Pred End',
               zorder=10, edgecolors='black', linewidths=1.5)
    te = np.sqrt((u_true[:, xi] - u_pred[:, xi]) ** 2 + (v_true[:, xi] - v_pred[:, xi]) ** 2).mean()
    ax.set_xlabel('u', fontweight='bold'); ax.set_ylabel('v', fontweight='bold')
    ax.set_title(f'x = {xi} (Traj. Error: {te:.3e})', fontweight='bold')
    ax.grid(True, alpha=0.3); ax.legend(loc='best', framealpha=0.9)
plt.tight_layout()
p2 = os.path.join(FIG_DIR, 'phase_portraits.png')
plt.savefig(p2, dpi=300, bbox_inches='tight'); plt.close(); print('saved', p2)

# Figure C: rollout error evolution
test_steps = [s for s in [1, 5, 10, 20, 30, 40, 50] if s <= N_STEPS]
n_eval = min(10, U.shape[0])
roll = {}
for ns in test_steps:
    um_ = {'rel_l2': [], 'mse': [], 'mae': [], 'max_ae': []}
    vm_ = {'rel_l2': [], 'mse': [], 'mae': [], 'max_ae': []}
    for si in range(n_eval):
        up, vp = _rollout(U[si, 0], V[si, 0], P[si], ns)
        ut = U[si, :ns + 1]; vt = V[si, :ns + 1]
        um_['rel_l2'].append(_relL2(up, ut)); um_['mse'].append(np.mean((up - ut) ** 2))
        um_['mae'].append(np.mean(np.abs(up - ut))); um_['max_ae'].append(np.max(np.abs(up - ut)))
        vm_['rel_l2'].append(_relL2(vp, vt)); vm_['mse'].append(np.mean((vp - vt) ** 2))
        vm_['mae'].append(np.mean(np.abs(vp - vt))); vm_['max_ae'].append(np.max(np.abs(vp - vt)))
    roll[ns] = {'u': {k: float(np.mean(um_[k])) for k in um_},
                'v': {k: float(np.mean(vm_[k])) for k in vm_}}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, (m, title) in enumerate(zip(['rel_l2', 'mse', 'mae', 'max_ae'],
                                     ['Relative L2 Error', 'MSE', 'MAE', 'Max Absolute Error'])):
    ax = axes[idx // 2, idx % 2]
    ax.plot(test_steps, [roll[s]['u'][m] for s in test_steps], color=COL['true'],
            lw=2, marker='o', ms=4, label='u (Activator)')
    ax.plot(test_steps, [roll[s]['v'][m] for s in test_steps], color=COL['pred'],
            lw=2, marker='s', ms=4, label='v (Inhibitor)')
    ax.set_xlabel('Rollout Step', fontweight='bold'); ax.set_ylabel(title, fontweight='bold')
    ax.set_title(f'{title} vs Rollout Length', fontweight='bold'); ax.legend(); ax.grid(True, alpha=0.3)
    if m == 'mse': ax.set_yscale('log')
plt.tight_layout()
p3 = os.path.join(FIG_DIR, 'rollout_error_evolution.png')
plt.savefig(p3, dpi=300, bbox_inches='tight'); plt.close(); print('saved', p3)

# dump the rollout numbers so captions stay consistent
import json as _json
with open(os.path.join(FIG_DIR, 'rollout_errors.json'), 'w') as f:
    _json.dump({str(k): v for k, v in roll.items()}, f, indent=2)
print('\nDONE. Download these from', FIG_DIR)
print('  single_step_regimes.png, phase_portraits.png, rollout_error_evolution.png')
print('  then drop them into paper figures/ (overwrite) and recompile.')
